In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
# 데이터 준비
# 설명변수
X_train = np.array([[0.0, 0.0],
                    [1.0, 0.0],
                    [0.0, 1.0],
                    [1.0, 1.0]], dtype=np.float32)

# 목표변수(XOR 연산)
y_train = np.array([[0.0],
                    [1.0],
                    [1.0],
                    [0.0]], dtype=np.float32)

In [3]:
# model 구성   
model = Sequential()
model.add(Input(shape=(2,)))                   # 입력층  2개
model.add(Dense(units=3, activation='tanh'))   # 은닉층 -> tanh (xor연산에 적합)
model.add(Dense(units=1, activation='sigmoid')) # 출력층 1개

In [4]:
# model compile
model.compile(loss='binary_crossentropy',    # 이진분류의 손실함수
              optimizer=RMSprop(learning_rate=0.01), # 학습률 가중치를 얼마나 크게 수정할지 결정
              metrics=['accuracy'])  # 정확도

model.summary()  # 모델 시각화  <- 확인용

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 3)                   │               9 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 13 (52.00 B)

 Trainable params: 13 (52.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# callback(option): 과적합방지 / 학습 자동 중단
early = EarlyStopping(monitor='loss', patience=100, restore_best_weights=True, verbose=0)

In [6]:
# 학습
history = model.fit(X_train, y_train,  # 입력데이터, 출력(정답)데이터
                    batch_size=4,      # 한번에 몇개의 데이터를 묶어서 학습할 것인가? 
                    epochs=2000,       # 전체의 모델의 학습을 반복하는 횟수
                    verbose=2,         # 0: 아무것도 출력안함 1: 진행  2:한줄 요약 로그출력
                    callbacks=[early]) # 학습하는 사이에 개입하는 감시자   

# 1/1 - 1s - 793ms/step            - accuracy: 0.5000 - loss: 0.6971
#            한 step에 걸리는 시간    정확도              손실함수 값

Epoch 1/2000
1/1 - 1s - 793ms/step - accuracy: 0.5000 - loss: 0.6971
Epoch 2/2000
1/1 - 0s - 39ms/step - accuracy: 0.7500 - loss: 0.6911
Epoch 3/2000
1/1 - 0s - 39ms/step - accuracy: 0.7500 - loss: 0.6885
Epoch 4/2000
1/1 - 0s - 38ms/step - accuracy: 0.7500 - loss: 0.6865
Epoch 5/2000
1/1 - 0s - 41ms/step - accuracy: 0.5000 - loss: 0.6847
Epoch 6/2000
1/1 - 0s - 39ms/step - accuracy: 0.5000 - loss: 0.6829
Epoch 7/2000
1/1 - 0s - 35ms/step - accuracy: 0.7500 - loss: 0.6811
Epoch 8/2000
1/1 - 0s - 32ms/step - accuracy: 0.7500 - loss: 0.6793
Epoch 9/2000
1/1 - 0s - 33ms/step - accuracy: 0.7500 - loss: 0.6774
Epoch 10/2000
1/1 - 0s - 35ms/step - accuracy: 1.0000 - loss: 0.6755
Epoch 11/2000
1/1 - 0s - 40ms/step - accuracy: 1.0000 - loss: 0.6736
Epoch 12/2000
1/1 - 0s - 42ms/step - accuracy: 1.0000 - loss: 0.6715
Epoch 13/2000
1/1 - 0s - 37ms/step - accuracy: 1.0000 - loss: 0.6694
Epoch 14/2000
1/1 - 0s - 44ms/step - accuracy: 1.0000 - loss: 0.6672
Epoch 15/2000
1/1 - 0s - 37ms/step - accur

In [8]:
# 평가 및 예측
model.make_predict_function()   # 예측 함수 생성

test_loss, test_acc = model.evaluate(X_train, y_train, verbose=0)
print('최종 학습 결과 -> loss: {:.6f}, accuracy(정확도): {:.6f}'.format(test_loss, test_acc))

최종 학습 결과 -> loss: 0.000057, accuracy(정확도): 1.000000


In [10]:
# predict(예측)
pred_y = model.predict(X_train)
y_train.reshape(-1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


array([0., 1., 1., 0.], dtype=float32)

In [16]:
np.set_printoptions(suppress=True)
pred_y.reshape(-1)

array([0.00000072, 0.99993163, 0.9999317 , 0.00009124], dtype=float32)

In [19]:
# 새로운 데이터로 모델을 이용해서 출력
test_data = np.array([[0.0, 1.0]], dtype=np.float32)

pred_value = model.predict(test_data)

binary_val = (pred_value > 0.5).astype(int)

print("입력:", test_data[0].tolist())
print("확률:", pred_value[0].item())
print("예측:", binary_val[0].item())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
입력: [0.0, 1.0]
확률: 0.9999316930770874
예측: 1
